In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import torch

from src.dataset.kitti_dataset import KittiDataset
from src.models.bev_detector import BEVDetector
from src.training.training_script import create_batch


In [ ]:
dataset = KittiDataset(PROJECT_ROOT / "data" / "KITTI", split="training")
sample = dataset[1]
bev, targets_np = dataset.get_training_sample(1)
print(bev.shape)


In [ ]:
bev_tensor = torch.from_numpy(
    bev
).float()
print(bev_tensor.shape)

In [ ]:
bev_tensor = bev_tensor.unsqueeze(0)

print(bev_tensor.shape)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

In [ ]:
model = BEVDetector().to(device)

bev_tensor = bev_tensor.to(device)

In [ ]:
model.eval()

with torch.no_grad():
    outputs = model(bev_tensor)

In [ ]:
for name, tensor in outputs.items():
    print(name, tensor.shape)

In [ ]:

from src.dataset.kitti_dataset import KittiDataset
from src.preprocessing.bev import bev_projection

from src.geometry.boxes import (
    create_box_corners_camera,
    box_parameters_from_lidar_corners,
    get_lidar_boxes
)

from src.geometry.transforms import (
    rectified_camera_to_lidar
)

from src.targets.bev_targets import (
    create_bev_targets
)

In [ ]:
points = sample["points"]
labels = sample["labels"]
calib = sample["calib"]

bev = bev_projection(points)

In [ ]:
boxes = get_lidar_boxes(
    labels,
    calib
)

In [ ]:
targets_np = create_bev_targets(boxes)

In [ ]:
import torch
target_tensors = {
    "heatmap": torch.from_numpy(
        targets_np["heatmap"]
    ).float().unsqueeze(0).unsqueeze(0),

    "offset": torch.from_numpy(
        targets_np["offset"]
    ).float().unsqueeze(0),

    "size": torch.from_numpy(
        targets_np["size"]
    ).float().unsqueeze(0),

    "rotation": torch.from_numpy(
        targets_np["rotation"]
    ).float().unsqueeze(0),

    "regression_mask": torch.from_numpy(
        targets_np["regression_mask"]
    ).float().unsqueeze(0).unsqueeze(0)
}

In [ ]:
target_tensors = {
    key: value.to(device)
    for key, value in target_tensors.items()
}

In [ ]:
from src.losses.detection_loss import DetectionLoss

criterion = DetectionLoss().to(device)

model.train()

predictions = model(bev_tensor)

losses = criterion(
    predictions,
    target_tensors
)

for name, value in losses.items():
    print(
        name,
        value.item()
    )

In [ ]:
print("mask sum:",
      target_tensors["regression_mask"].sum().item())

print("heatmap positives:",
      target_tensors["heatmap"].sum().item())

In [ ]:
import numpy as np
targets_np = create_bev_targets(boxes)

print(
    "heatmap sum:",
    targets_np["heatmap"].sum()
)

print(
    "regression mask sum:",
    targets_np["regression_mask"].sum()
)

print(
    "non-zero offset values:",
    np.count_nonzero(targets_np["offset"])
)

print(
    "non-zero size values:",
    np.count_nonzero(targets_np["size"])
)

print(
    "non-zero rotation values:",
    np.count_nonzero(targets_np["rotation"])
)

In [ ]:
indices = np.argwhere(
    targets_np["regression_mask"] > 0
)

print(indices)

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

history = []

model.train()

for step in range(5):

    optimizer.zero_grad()

    predictions = model(bev_tensor)

    losses = criterion(
        predictions,
        target_tensors
    )

    loss = losses["total"]

    loss.backward()

    optimizer.step()

    history.append({
        "total": loss.item(),
        "heatmap": losses["heatmap"].item(),
        "offset": losses["offset"].item(),
        "size": losses["size"].item(),
        "rotation": losses["rotation"].item()
    })

    if step % 20 == 0:
        print(
            f"step {step:03d} | "
            f"total {loss.item():.4f} | "
            f"heatmap {losses['heatmap'].item():.4f} | "
            f"offset {losses['offset'].item():.4f} | "
            f"size {losses['size'].item():.4f} | "
            f"rotation {losses['rotation'].item():.4f}"
        )

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.plot(
    [x["total"] for x in history],
    label="total"
)

plt.plot(
    [x["heatmap"] for x in history],
    label="heatmap"
)

plt.plot(
    [x["offset"] for x in history],
    label="offset"
)

plt.plot(
    [x["size"] for x in history],
    label="size"
)

plt.plot(
    [x["rotation"] for x in history],
    label="rotation"
)

plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("Single-sample overfit test")
plt.legend()

plt.show()

In [ ]:
from src.dataset.kitti_dataset import KittiDataset

dataset = KittiDataset(PROJECT_ROOT / "data" / "KITTI", split="training")

bev, targets = dataset.get_training_sample(10)

print(bev.shape)
print(targets["heatmap"].shape)
print(targets["offset"].shape)
print(targets["size"].shape)
print(targets["rotation"].shape)
print(targets["regression_mask"].shape)

In [ ]:
# Batch construction uses src.training.training_script.create_batch.


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

bev_batch, targets_batch = create_batch(
    dataset,
    indices=[1, 2, 3, 4],
    device=device
)

print("BEV:", bev_batch.shape)

for name, tensor in targets_batch.items():
    print(name, tensor.shape)

In [ ]:
print(dataset.velodyne_dir)
print(dataset.velodyne_dir.exists())
print(len(dataset))

In [ ]:
dataset = KittiDataset(PROJECT_ROOT / "data" / "KITTI", split="training")
print("Dataset length:", len(dataset))
print("First IDs:", dataset.sample_ids[:5])
print("Last IDs:", dataset.sample_ids[-5:])


In [ ]:
# Use the import-safe command-line trainer for full runs:
# python -m src.training.training_script --data-root data/KITTI --epochs 30
print("Full training is intentionally not launched from this notebook.")
